# Stage1 Underperformer Retention Cleanup V1

目的：保护当前 Stage1，删除三条明确未晋升的独立可运行链，只把可复用经验和删除清单保留在 qlib Recorder 与 slim card 中。

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not ((ROOT / 'AGENTS.md').exists() and (ROOT / 'SPEC.md').exists()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import qlib
from qlib.data import D
from qlib.workflow import R

EXPERIMENT = 'stage1_underperformer_retention_cleanup_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER = ROOT / 'data' / 'qlib' / 'soxl_soxx_soxs'
qlib.init(provider_uri=PROVIDER.as_posix(), region='us', kernels=1)
calendar = D.calendar(freq='day')
{'provider_uri': PROVIDER.as_posix(), 'calendar_start': str(calendar[0]), 'calendar_end': str(calendar[-1])}

[31687:MainThread](2026-07-25 22:13:22,150) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31687:MainThread](2026-07-25 22:13:22,151) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31687:MainThread](2026-07-25 22:13:22,151) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs')}


{'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs',
 'calendar_start': '2010-03-11 00:00:00',
 'calendar_end': '2026-07-24 00:00:00'}

In [2]:
decisions = pd.read_csv(CARD / 'cleanup_decisions.csv')
delete_manifest = pd.read_csv(CARD / 'delete_manifest.csv')
retained_ideas = (CARD / 'retained_ideas.md').read_text(encoding='utf-8')
baseline = json.loads((ROOT / 'research' / 'stage1_generalization_validation_v1' / 'summary.json').read_text(encoding='utf-8'))

protected_paths = [
    'research/pit_constituent_auxiliary_training_allocation_v1',
    'research/intraday_recovery_direct_allocation_v1',
    'research/intraday_on_policy_staged_validation_v1',
    'research/stage1_generalization_validation_v1',
    'production/stage1_futu_sim_v1',
    'soxl_backtest',
]
missing_protected = [path for path in protected_paths if not (ROOT / path).exists()]
remaining_deleted = [path for path in delete_manifest['path'] if (ROOT / path).exists()]
if missing_protected:
    raise RuntimeError(f'blocked: protected Stage1 surfaces are missing: {missing_protected}')
if remaining_deleted:
    raise RuntimeError(f'blocked: cleanup manifest still exists: {remaining_deleted}')

recorder_uri = CARD / 'qlib_recorder'
recorder_uri.mkdir(parents=True, exist_ok=True)
R.set_uri(recorder_uri.resolve().as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    R.log_params(
        protected_baseline='stage1',
        replacement_promoted='false',
        selection_uses_outer_test='false',
        deleted_runnable_chains='3',
    )
    R.log_metrics(
        stage1_outer_return_pct=float(baseline['frozen_stage1_metrics_10bps']['return_pct']),
        stage1_outer_max_drawdown_pct=float(baseline['frozen_stage1_metrics_10bps']['max_drawdown_pct']),
        deleted_runnable_chains=3.0,
    )
    for filename in ('cleanup_decisions.csv', 'delete_manifest.csv', 'retained_ideas.md'):
        recorder.log_artifact((CARD / filename).as_posix(), artifact_path=EXPERIMENT)
    recorder_result = {
        'recorder_id': recorder.id,
        'recorder_uri': recorder_uri.resolve().as_posix(),
        'recorder_local_dir': Path(recorder.get_local_dir()).resolve().as_posix(),
    }

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'stage1_underperformer_retention_cleanup',
    'status': 'cleanup_complete_stage1_protected_no_replacement_promoted',
    'decision': 'retain_stage1_delete_three_underperforming_runnable_chains',
    'replacement_promoted': False,
    'selection_uses_outer_test': False,
    'outer_role': 'frozen_diagnostic_only_not_model_selection',
    'system_of_record': 'qlib',
    'provider_uri': PROVIDER.resolve().as_posix(),
    'market': 'all',
    'region': 'us',
    'as_of': str(pd.Timestamp(calendar[-1]).date()),
    'protected_paths': protected_paths,
    'deleted_runnable_chains': 3,
    'deleted_manifest_entries': int(len(delete_manifest)),
    'stage1_metrics_10bps': baseline['frozen_stage1_metrics_10bps'],
    'source_validation': 'research/stage1_generalization_validation_v1',
    'source_stage1_recorder_id': baseline['source_recorder_id'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
        'strategy': 'https://qlib.readthedocs.io/en/latest/component/strategy.html',
    },
    'sidecars': {
        'cleanup_decisions': 'cleanup_decisions.csv',
        'delete_manifest': 'delete_manifest.csv',
        'retained_ideas': 'retained_ideas.md',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'recorder_only_artifacts': [],
    },
    **recorder_result,
}
(CARD / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

metrics = summary['stage1_metrics_10bps']
conclusion = f'''# 中文结论

- 本轮没有找到满足稳健替换门槛的新模型，`replacement_promoted=false`；Stage1 继续作为受保护基线。
- 当前 Stage1 frozen outer 10bp：收益 {metrics['return_pct']:.2f}%，MaxDD {metrics['max_drawdown_pct']:.2f}%，最差五日 {metrics['worst_rolling_5d_pct']:.2f}%。
- 已删除三条未晋升的独立可运行链：成本感知目标跟踪、GRU 状态直接分配、Stage1 扩展特征恢复；对应源代码、专属测试和旧研究卡均进入删除清单。
- 删除不等于遗忘：核心指标、失败机制、重启条件和本轮结构诊断已保存在 `cleanup_decisions.csv` 与 `retained_ideas.md`，并写入 qlib Recorder。
- outer 没有参与选择或反向调参；本卡只记录冻结诊断和清理决定。
- 受保护范围包括 Stage1 来源模型、恢复动作、on-policy 验证、泛化验证、富途模拟盘生产卡与 `soxl_backtest` legacy 对照。
- qlib recorder：{recorder_result['recorder_id']}
- 下一步：只有新增 PIT 训练数据、成交成本标签或预登记 forward holdout 时，才重新开启已删除方向。
- 不构成买卖建议。
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
summary

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[31687:MainThread](2026-07-25 22:13:22,171) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_underperformer_retention_cleanup_v1.


[31687:MainThread](2026-07-25 22:13:22,173) INFO - qlib.workflow - [exp.py:258] - Experiment 577638251397202674 starts running ...


[31687:MainThread](2026-07-25 22:13:22,229) INFO - qlib.workflow - [recorder.py:345] - Recorder 7ea48e01114c493c948f9a660dceea09 starts running under Experiment 577638251397202674 ...


[31687:MainThread](2026-07-25 22:13:22,343) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{'experiment': 'stage1_underperformer_retention_cleanup_v1',
 'card_type': 'stage1_underperformer_retention_cleanup',
 'status': 'cleanup_complete_stage1_protected_no_replacement_promoted',
 'decision': 'retain_stage1_delete_three_underperforming_runnable_chains',
 'replacement_promoted': False,
 'selection_uses_outer_test': False,
 'outer_role': 'frozen_diagnostic_only_not_model_selection',
 'system_of_record': 'qlib',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs',
 'market': 'all',
 'region': 'us',
 'as_of': '2026-07-24',
 'protected_paths': ['research/pit_constituent_auxiliary_training_allocation_v1',
  'research/intraday_recovery_direct_allocation_v1',
  'research/intraday_on_policy_staged_validation_v1',
  'research/stage1_generalization_validation_v1',
  'production/stage1_futu_sim_v1',
  'soxl_backtest'],
 'deleted_runnable_chains': 3,
 'deleted_manifest_entries': 9,
 'stage1_metrics_10bps': {'terminal_wealth': 6.4178894962121875,
  'return_pct': 541.7

In [3]:
from IPython.display import Markdown, display

display(Markdown((CARD / 'conclusion.md').read_text(encoding='utf-8')))

# 中文结论

- 本轮没有找到满足稳健替换门槛的新模型，`replacement_promoted=false`；Stage1 继续作为受保护基线。
- 当前 Stage1 frozen outer 10bp：收益 541.79%，MaxDD -28.11%，最差五日 -11.60%。
- 已删除三条未晋升的独立可运行链：成本感知目标跟踪、GRU 状态直接分配、Stage1 扩展特征恢复；对应源代码、专属测试和旧研究卡均进入删除清单。
- 删除不等于遗忘：核心指标、失败机制、重启条件和本轮结构诊断已保存在 `cleanup_decisions.csv` 与 `retained_ideas.md`，并写入 qlib Recorder。
- outer 没有参与选择或反向调参；本卡只记录冻结诊断和清理决定。
- 受保护范围包括 Stage1 来源模型、恢复动作、on-policy 验证、泛化验证、富途模拟盘生产卡与 `soxl_backtest` legacy 对照。
- qlib recorder：7ea48e01114c493c948f9a660dceea09
- 下一步：只有新增 PIT 训练数据、成交成本标签或预登记 forward holdout 时，才重新开启已删除方向。
- 不构成买卖建议。
